## Requirements

In [ ]:
import os, sys
import random
import joblib # save and load objects
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ParameterGrid, TimeSeriesSplit


In [ ]:
# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") #automatically runs your model on GPU if available, otherwise CPU
# print("Using device:", DEVICE)

Using device: cpu


In [ ]:
# seed = 42
# torch.manual_seed(seed)
# torch.cuda.manual_seed_all(seed)
# np.random.seed(seed)
# random.seed(seed)

# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = False

In [201]:
src_path = os.path.join(os.getcwd(), "src")
if src_path not in sys.path:
    sys.path.append(src_path)

from data_pipeline import MirrorLogNormScaler
from model_evaluation import evaluate_model

In [202]:
# import os
# print(os.getcwd())

In [203]:
# df = pd.read_csv("./data/raw/dataset_2019_2025.csv", parse_dates=["datetime"], index_col="datetime")
# print(df.shape)
# df.head()

In [220]:
df = pd.read_csv("./data/processed/rfe_dataset_2019_2025.csv", parse_dates=["datetime"], index_col="datetime")
print(df.shape)
df.head()
df.tail()
# df.size

(52560, 30)


,price,temperature_24,temperature_28,temperature_33,temperature_40,temperature_46,relative_humidity_29,relative_humidity_36,relative_humidity_48,load_24,...,sum_cbet_43,sum_cbet_48,price_24,price_31,price_37,price_41,price_43,price_45,price_47,price_48
datetime,,,,,,,,,,,,,,,,,,,,,
2024-12-31 19:00:00+00:00,35.56,1.050,1.700,1.475,1.000,0.675,91.25,96.00,96.00,55088.7,...,-5.767,-5.446,74.83,80.50,80.20,64.65,68.60,71.63,90.21,109.92
2024-12-31 20:00:00+00:00,15.70,0.825,1.625,1.750,1.025,0.700,91.25,95.50,96.25,53349.1,...,-3.773,-5.374,68.02,83.70,86.30,66.89,68.60,71.73,83.79,90.21
2024-12-31 21:00:00+00:00,9.06,0.700,1.575,1.850,0.950,0.750,92.00,94.75,96.75,50345.0,...,-3.474,-6.301,67.26,92.42,87.06,69.17,64.65,68.60,71.63,83.79
2024-12-31 22:00:00+00:00,0.52,0.575,1.500,1.950,0.950,0.750,91.50,94.25,96.75,46199.6,...,-4.771,-5.709,35.66,96.97,83.71,71.14,66.89,68.60,71.73,71.63
2024-12-31 23:00:00+00:00,2.16,0.475,1.050,1.975,1.125,0.875,90.75,94.00,96.75,44047.5,...,-6.397,-5.458,50.49,109.97,78.88,80.20,69.17,64.65,68.60,71.73


In [205]:
# df = df.loc["2019-01-03" : "2019-12-31"]
# df

In [206]:
# Define X (features) and y(price)
y_all = df["price"].astype(float)
X_all = df.drop(columns=["price"]).astype(float)
# y_all

## Split Training/Testing set

In [ ]:
# Split 90% train+validation, 10% test
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, shuffle=False) # equivalent

N = len(df)
n_test = int(np.ceil(0.10 * N))
n_trainval = N - n_test

X_trainval = X_all.iloc[:n_trainval].copy()
y_trainval = y_all.iloc[:n_trainval].copy()
X_test = X_all.iloc[n_trainval:].copy()
y_test = y_all.iloc[n_trainval:].copy()

print(f"Total samples: {N}")
print(f"Train+Val: {len(X_trainval)} ({len(X_trainval)/N:.1%})")
print(f"Test: {len(X_test)} ({len(X_test)/N:.1%})")

Total samples: 52560
Train+Val: 47304 (90.0%)
Test: 5256 (10.0%)


## Scaling

In [ ]:
x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_trainval_scaled = x_scaler.fit_transform(X_trainval)
y_trainval_scaled = y_scaler.fit_transform(y_trainval.to_numpy().reshape(-1, 1)).reshape(-1)

X_test_scaled = x_scaler.transform(X_test)
y_test_scaled = y_scaler.transform(y_test.to_numpy().reshape(-1, 1)).reshape(-1) # change the price col into a 2D array, turns it back after scaling

print("X_trainval:", X_trainval_scaled.shape, "  X_test:", X_test_scaled.shape)

X_trainval: (47304, 29)   X_test: (5256, 29)


## Build a Dataset that yields LSTM-ready sequences

In the dataset, each row = 1 timestamp, each column = a feature

LSTM need sequences of consecutive time steps (past 24h of features to predict the next hour's price)

Take the long time series and cut it into many overlapping windows, each window becomes one training sample for the LSTM.

Index:0, Input sequence: times 0-23, Predict: time 23

Index:1, Input sequence: times 1-24, Predict: time 24

...

In [ ]:
# Sequence dataset + a tiny loader helper 
class SequenceDataset(Dataset):
    """
    Turn a time-ordered table into overlapping sequences for LSTM.

    Inputs:
      X_array: np.ndarray of shape (T, n_features)   — scaled features
      y_array: np.ndarray of shape (T,)              — scaled target
      seq_len: int                                   — lookback window length

    Output per item (for index i):
      x_seq: torch.FloatTensor of shape (seq_len, n_features)
      y_t:   torch.FloatTensor scalar — the target at the *next* step of the window
    """
    def __init__(self, X_array, y_array, seq_len: int):
        X_array = np.asarray(X_array, dtype=np.float32) #2D
        y_array = np.asarray(y_array, dtype=np.float32).reshape(-1) #1d vector

        assert X_array.ndim == 2, "X must be 2D: (T, n_features)"
        assert y_array.ndim == 1, "y must be 1D: (T,)"
        assert len(X_array) == len(y_array), "X and y must align along time"

        self.X = X_array
        self.y = y_array
        self.seq_len = int(seq_len) # how many past time steps to use

        # calculate how many sequences can we form?
        # For T rows, the valid start indices are 0 .. T - seq_len
        self.length = len(self.y) - self.seq_len
        if self.length <= 0:
            raise ValueError(f"seq_len={seq_len} is longer than the series length={len(self.y)}")

    # how many sliding windows exist
    def __len__(self):
        return self.length

    def __getitem__(self, idx: int):
        # Build the window [idx, idx+seq_len)
        x_seq = self.X[idx : idx + self.seq_len]                 # (seq_len, n_features)
        # Predict the value at the *last* time step in this window
        y_t   = self.y[idx + self.seq_len]   # NEW, predict the next hour                # scalar
        return torch.from_numpy(x_seq), torch.tensor(y_t, dtype=torch.float32)

# Packages everything into a DataLoader that handles batching and feeding data into the model.
def make_loader(X, y, seq_len: int, batch_size: int, shuffle: bool = False):
    """
    Convenience wrapper to build a DataLoader from arrays.
    """
    ds = SequenceDataset(X, y, seq_len)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)


## LSTM Model Define

In [ ]:
#  Define the LSTM model 

class LSTMRegressor(nn.Module):
    """
    A simple LSTM-based regression model for one-step-ahead forecasting.

    Input shape:  (batch, seq_len, n_features)
    Output shape: (batch,)   → predicted price (scaled)
    """
    def __init__(self, n_features: int,
                 hidden_size: int = 64, # how many neurons per LSTM layer, controls how much the model can learn (bigger = more capacity)
                 num_layers: int = 2,  # how many LSTM layers are stacked (deeper = learns more complex patterns)
                 dropout: float = 0.2): #prevents overfitting by randomly turning off some neurons between layers
        super().__init__()

        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=n_features, # number of features at each time step (29 columns).
            hidden_size=hidden_size, # number of neurons in each LSTM layer (controls capacity)
            num_layers=num_layers, # how many stacked LSTMs use.
            batch_first=True,     # input/output tensors follow the (batch, seq_len, features) convention (matches DataLoader).
            dropout=dropout if num_layers > 1 else 0.0 # adds regularization between stacked layers (ignored if only one layer)
        )

        # Final linear (fully connected) layer, turns it into a single predicted value.
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        """
        Forward pass:
        x : (batch, seq_len, n_features)
        returns (batch,)  — predicted value
        """
        # out: (batch, seq_len, hidden_size), the hidden state for every time step
        # h_n: (num_layers, batch, hidden_size), the final hidden state for each layer
        # c_n: (num_layers, batch, hidden_size), the final cell state
        out, (h_n, c_n) = self.lstm(x)

        # Take the last time step's hidden output from the top LSTM layer
        last_hidden = out[:, -1, :]          # the output at the final time step, encodes everything the LSTM learned about the past seq_len points, (batch, hidden_size)

        # Pass through a linear head to get 1 value per sample
        y_pred = self.fc(last_hidden)        # Converts the hidden_size vector into one scalar (the predicted price), (batch, 1)
        return y_pred.squeeze(-1)            # (batch,)


## Training loop with early stopping

Train one LSTM on the training slice, watch how it performs on the validation slice, and stop early when validation stops improving. Prevent overfitting. Then return the best model plus the loss history.

In [ ]:

def train_one_model(X_tr, y_tr,
                    X_va, y_va,
                    seq_len: int,
                    model_params: dict,
                    max_epochs: int = 60,
                    patience: int = 8):
    """
    Train a single LSTM model on (train, validation) data.

    Args:
        X_tr, y_tr: scaled training arrays
        X_va, y_va: scaled validation arrays
        seq_len: lookback window
        model_params: dict of LSTM hyperparameters
        max_epochs: max training epochs
        patience: stop if val loss doesn't improve after this many epochs
    """
    n_features = X_tr.shape[1]
    model = LSTMRegressor(
        n_features=n_features,
        hidden_size=model_params.get("hidden_size", 64),
        num_layers=model_params.get("num_layers", 2),
        dropout=model_params.get("dropout", 0.2)
    ).to(DEVICE)

    # Build DataLoaders
    train_loader = make_loader(X_tr, y_tr, seq_len, batch_size=model_params.get("batch_size", 128))
    val_loader   = make_loader(X_va, y_va, seq_len, batch_size=model_params.get("batch_size", 128)) 
    # Loss + optimizer
    criterion = nn.MSELoss() # smaller is better
    optimizer = torch.optim.Adam(model.parameters(), lr=model_params.get("lr", 1e-3)) # Adam. It tweaks the model’s weights to reduce the loss.

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    train_losses, val_losses = [], []

# training loop
    for epoch in range(1, max_epochs + 1):
        # TRAIN 
        model.train()
        epoch_train_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            y_pred = model(xb) # forward pass
            loss = criterion(y_pred, yb)
            loss.backward() # compute gradients
            optimizer.step()  # update weights
            epoch_train_loss += loss.item() * len(xb)

        epoch_train_loss /= len(train_loader.dataset)
        train_losses.append(epoch_train_loss)

        # VALIDATION 
        model.eval() # switches off training-only behaviors
        epoch_val_loss = 0
        with torch.no_grad(): # turns off gradient tracking to save memory/speed
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                y_pred = model(xb)
                loss = criterion(y_pred, yb)
                epoch_val_loss += loss.item() * len(xb)

        epoch_val_loss /= len(val_loader.dataset)
        val_losses.append(epoch_val_loss) #compute the average validation loss without updating weights.

        print(f"Epoch {epoch:03d} | train={epoch_train_loss:.6f}  val={epoch_val_loss:.6f}")

        # Early stopping check
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    # Load the best model weights
    if best_state is not None:
        model.load_state_dict(best_state)

    return model, train_losses, val_losses


In [212]:
# params_demo = {
#     "hidden_size": 64,
#     "num_layers": 2,
#     "dropout": 0.2,
#     "lr": 1e-3,
#     "batch_size": 128
# }

# # Split 10% of trainval as validation (simple example)
# split_i = int(len(X_trainval_scaled) * 0.8)
# X_tr, X_va = X_trainval_scaled[:split_i], X_trainval_scaled[split_i:]
# y_tr, y_va = y_trainval_scaled[:split_i], y_trainval_scaled[split_i:]

# model, tr_loss, va_loss = train_one_model(X_tr, y_tr, X_va, y_va,
#                                           seq_len=24,
#                                           model_params=params_demo,
#                                           max_epochs=20,
#                                           patience=5)


## Rolling Window Grid Search

In [ ]:
# Rolling-Window Hyperparameter Tuning
# Define a grid of hyperparameters to search over
param_grid = {
    "hidden_size": [32, 64],
    "num_layers": [1, 2],
    "dropout": [0.1, 0.2],
    "lr": [1e-3, 5e-4],
    "batch_size": [64, 128] # how much data feed to the model at one time, smaller batch size, more frequent updates, possibly better generalization
}

# Rolling-window cross-validation function
def rolling_window_cv(X, y, n_splits=3, test_size=0.1, seq_len=24, param_grid=None):
    """
    Perform time-based (rolling) cross-validation for LSTM hyperparameter tuning.
    Each fold trains on past data and validates on future data.
    """

    N = len(X)

    # last 10% as final test set
    n_test = int(0.10 * N)
    X_trainval, y_trainval = X[:-n_test], y[:-n_test]
    print(f"Train+Val samples: {len(X_trainval)} | Test samples (held-out): {n_test}")

    results = []

    # Inside the 90%, perform 3 rolling folds
    # Each fold uses 80% of trainval for training and 20% for validation
    train_ratio = 0.8
    fold_size = int((1 - train_ratio) * len(X_trainval) / n_splits)
    train_size = int(train_ratio * len(X_trainval))

    for params in ParameterGrid(param_grid):
        val_losses = []
        print(f"\n--- Testing params: {params} ---")

        for i in range(n_splits):
            # define rolling window indices
            train_end = train_size + i * fold_size
            val_start = train_end
            val_end = val_start + fold_size

            if val_end > len(X_trainval):
                break  # prevent overflow on last fold

            X_train = X_trainval[:train_end]
            y_train = y_trainval[:train_end]
            X_val = X_trainval[val_start:val_end]
            y_val = y_trainval[val_start:val_end]

            model, tr_loss, va_loss = train_one_model(
                X_train, y_train, X_val, y_val,
                seq_len=seq_len,
                model_params=params,
                max_epochs=40,
                patience=6
            )

            # store final validation loss from this fold
            val_losses.append(va_loss[-1])
            print(f"Fold {i+1}/{n_splits} → final val_loss = {va_loss[-1]:.6f}")

        avg_val_loss = np.mean(val_losses)
        results.append((params, avg_val_loss))
        print(f"Average val_loss for params: {avg_val_loss:.6f}")

    # Pick best params with smallest validation loss
    best_params, best_val_loss = min(results, key=lambda x: x[1])
    print("\n Best Params:", best_params)
    print(f"Best Validation Loss: {best_val_loss:.6f}")

    return best_params




    # n_samples = len(X)
    # fold_size = int(n_samples * test_size)
    # train_end = n_samples - n_splits * fold_size

    # results = []  # store (params, avg_val_loss)

    # # Try each parameter combination
    # for params in ParameterGrid(param_grid):
    #     val_losses = []

    #     print(f"\n--- Testing params: {params} ---")

    #     # Rolling folds
    #     for i in range(n_splits):
    #         start = i * fold_size
    #         end = train_end + i * fold_size

    #         X_train = X[:end]
    #         y_train = y[:end]
    #         X_val = X[end:end + fold_size]
    #         y_val = y[end:end + fold_size]

    #         # Train model on this fold
    #         model, tr_loss, va_loss = train_one_model(
    #             X_train, y_train, X_val, y_val,
    #             seq_len=seq_len,
    #             model_params=params,
    #             max_epochs=40,
    #             patience=6
    #         )
    #         val_losses.append(va_loss[-1])

    #         print(f"Fold {i+1}/{n_splits} -> final val_loss = {va_loss[-1]:.6f}")


    #     avg_val_loss = np.mean(val_losses)
    #     results.append((params, avg_val_loss))
    #     print(f"Average val_loss for params: {avg_val_loss:.6f}")

    # # Select the best-performing hyperparameters
    # best_params, best_val_loss = min(results, key=lambda x: x[1])
    # print("\nBest Params:", best_params)
    # print("Best Validation Loss:", best_val_loss)

    # return best_params


In [214]:
best_params = rolling_window_cv(
    X_trainval_scaled,
    y_trainval_scaled,
    n_splits=3,
    test_size=0.1,
    seq_len=24,
    param_grid=param_grid
)


Train+Val samples: 42574 | Test samples (held-out): 4730

--- Testing params: {'batch_size': 64, 'dropout': 0.1, 'hidden_size': 32, 'lr': 0.001, 'num_layers': 1} ---
Epoch 001 | train=0.003090  val=0.006716
Epoch 002 | train=0.001848  val=0.007598
Epoch 003 | train=0.001842  val=0.008775
Epoch 004 | train=0.001841  val=0.009887
Epoch 005 | train=0.001843  val=0.010782
Epoch 006 | train=0.001843  val=0.011290
Epoch 007 | train=0.001851  val=0.011682
Early stopping at epoch 7
Fold 1/3 → final val_loss = 0.011682
Epoch 001 | train=0.006252  val=0.003354
Epoch 002 | train=0.002360  val=0.003205
Epoch 003 | train=0.002345  val=0.002497
Epoch 004 | train=0.002344  val=0.001663
Epoch 005 | train=0.002336  val=0.001300
Epoch 006 | train=0.002324  val=0.001122
Epoch 007 | train=0.002313  val=0.001034
Epoch 008 | train=0.002299  val=0.001005
Epoch 009 | train=0.002292  val=0.000978
Epoch 010 | train=0.002274  val=0.000984
Epoch 011 | train=0.002253  val=0.000991
Epoch 012 | train=0.002234  val=0

## Final training & Evaluation RMSE

In [ ]:
# Final Training on Full Train+Val and Evaluation on Test

final_model, train_losses, val_losses = train_one_model(
    X_trainval_scaled, y_trainval_scaled,
    X_test_scaled, y_test_scaled,        
    seq_len=24,
    model_params=best_params,
    max_epochs=60,
    patience=8
)



# Plot training vs validation loss curve
# plt.figure(figsize=(7,4))
# plt.plot(train_losses, label="Train Loss")
# plt.plot(val_losses, label="Val/Test Loss")
# plt.xlabel("Epoch")
# plt.ylabel("MSE Loss")
# plt.title("Final Model Training Curve")
# plt.legend()
# plt.show()

# Make predictions on the test set
# Build test DataLoader
test_loader = make_loader(X_test_scaled, y_test_scaled,
                          seq_len=24, batch_size=best_params["batch_size"])

final_model.eval()
preds, actuals = [], []

with torch.no_grad(): #stops gradient tracking, making faster & saves memory during evaluation
    for xb, yb in test_loader:
        xb = xb.to(torch.float32)
        out = final_model(xb) #give predict prices
        preds.extend(out.cpu().numpy().flatten())
        actuals.extend(yb.cpu().numpy().flatten())

preds = np.array(preds)
actuals = np.array(actuals)

# Inverse-transform to real prices
y_pred_inv = y_scaler.inverse_transform(preds.reshape(-1, 1)).flatten()
y_true_inv = y_scaler.inverse_transform(actuals.reshape(-1, 1)).flatten()

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

rmse = np.sqrt(mean_squared_error(y_true_inv, y_pred_inv))
# mae = mean_absolute_error(y_true_inv, y_pred_inv)
# r2 = r2_score(y_true_inv, y_pred_inv)

print("\nFinal LSTM Test Evaluation:")
print(f"RMSE: {rmse:.4f}")
# print(f"MAE : {mae:.4f}")
# print(f"R²  : {r2:.4f}")

Epoch 001 | train=0.003535  val=0.002162
Epoch 002 | train=0.002400  val=0.002084
Epoch 003 | train=0.002300  val=0.001985
Epoch 004 | train=0.002163  val=0.001929
Epoch 005 | train=0.002079  val=0.001907
Epoch 006 | train=0.001998  val=0.001883
Epoch 007 | train=0.001938  val=0.001881
Epoch 008 | train=0.001880  val=0.001869
Epoch 009 | train=0.001827  val=0.001861
Epoch 010 | train=0.001775  val=0.001861
Epoch 011 | train=0.001730  val=0.001833
Epoch 012 | train=0.001680  val=0.001821
Epoch 013 | train=0.001640  val=0.001815
Epoch 014 | train=0.001601  val=0.001795
Epoch 015 | train=0.001562  val=0.001778
Epoch 016 | train=0.001522  val=0.001771
Epoch 017 | train=0.001492  val=0.001733
Epoch 018 | train=0.001455  val=0.001712
Epoch 019 | train=0.001425  val=0.001693
Epoch 020 | train=0.001395  val=0.001669
Epoch 021 | train=0.001366  val=0.001644
Epoch 022 | train=0.001333  val=0.001622
Epoch 023 | train=0.001313  val=0.001598
Epoch 024 | train=0.001284  val=0.001591
Epoch 025 | trai

In [ ]:
model_dir = "./models"
os.makedirs(model_dir, exist_ok=True)   # create folder if it doesn’t exist

# Save the trained LSTM model
model_path = os.path.join(model_dir, "lstm_5years_model.pth")
torch.save(final_model.state_dict(), model_path)
print(f"Model saved successfully to: {model_path}")

# Save the fitted scalers (can inverse transform predictions later)
x_scaler_path = os.path.join(model_dir, "x_scaler_5year.pkl")
y_scaler_path = os.path.join(model_dir, "y_scaler_5year.pkl")

joblib.dump(x_scaler, x_scaler_path)
joblib.dump(y_scaler, y_scaler_path)
print(f"Scalers saved successfully to: {x_scaler_path}, {y_scaler_path}")

✅ Model saved successfully to: ./models/lstm_5years_model.pth
✅ Scalers saved successfully to: ./models/x_scaler_5year.pkl, ./models/y_scaler_5year.pkl
